In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

In [2]:
## Load data from pickled files
df = pd.read_pickle("neg_sent.pkl")
df.head()

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,CMT_SENT,KEY_PHRASES
28,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,One of the most overrated games Ive ever heard...,8.5,269,-0.7727,"[overrated games ive ever heard, slightly modi..."
33,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,Good but as it grows it gets too chaotic addin...,8.5,135,-0.5187,"[streamlined experience, many rules, chaotic a..."
37,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,DIFFICULTY MEDIUM,8.5,18,-0.3400,[difficulty medium]
38,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,Bonn\nSob\nSchutte,8.5,19,-0.2500,[bonn sob schutte]
79,535d74a6-b776-48d8-93ab-712be79763c7,Pandemic Legacy: Season 1,Apart from one or two good twists very generic...,8.5,172,-0.0972,"[high luck factor card texts cannot, properly ..."


In [30]:
## Remove common noice in natural language
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 50:
        return text
    
## Create list of documents (comments)
documents = np.array(df['COMMENT'])

cleaned_docs = np.vectorize(clean_text)(documents)
cleaned_docs = np.array([x for x in cleaned_docs if x != 'None'])


In [119]:
custom_stopwords = list(ENGLISH_STOP_WORDS.union({
    "i", "favorite", "in", "game", "great", "good", "fun", "like", "really", "best", "ive", "games",
    "love", "absolutely", "just", "playing", "players", "gloomhaven", "pandemic", "played", "times",
    "far", "legacy", "coop", "amazing", "simply", "legacy", "better", "lot", "dont", "new", "experience",
    "awesome", "gaming", "fantastic", "season", "need", "want", "wait", "people", "friends", "different",
    "player", "excellent", "based", "rating", "perfect", "close", "introduction", "pretty", "set", "im",
    "spirits", "way", "group", "takes", "bit", "handed", "enjoy", "exclusively", "wife", "wish", "copy",
    "willing", "week", "worth", "table", "second", "gets", "having", "probably", "experiences", "recommend",
    "greatest", "truly", "enjoyable", "got", "bought", "usually", "able", "highly", "dd", "favourite", "rpg",
    "recommended", "date", "gamers", "blast", "getting", "think", "id", "family", "buy", "hard", "regular",
    "regularly", "works", "chance", "rated", "wanting", "difficult", "took", "know", "started", "right", "did",
    "excited", "wanted", "day", "waiting", "sure", "days", "year", "years", "joy", "higher", "review", "cards", 
    "board", "plays", "feel", "play", "true", "brain", "arkham", "horror", "que", "la", "el", "es", "en", "juego",
    "lo", "los", "una", "para", "carcosa", "dunwich", "return", "forgotten", "die", "und", "der", "das", "ist", "zu",
    "spiel", "aber", "ein", "jaws", "includes", "las", "se", "muy", "pero", "mas", "si", "por", "al", "te", "sin", 
    "rougarou", "curse", "make", "bad", "broken", "madness", "remnants", "forsaken", "man", "ich", "nicht", "sehr",
    "auch", "mit", "den", "fur", "eine", "auf", "earth", "circle", "circles", "age", "path", "investigator", "strange",
    "carnevale", "conspiracy", "innsmouth", "horrors", "zealot", "guardians", "murder", "excelsior", "dreameaters", "hotel",
    "abyss", "blob", "night", "ate", "keys", "scarlet", "lunacy", "investigators", "hemlock", "vale", "deck", "feast", "cho",
    "stella", "gods", "outer", "labyrinths", "nathaniel", "habbamock", "war", "clark", "harvey", "walters", "jacqueline",
    "fine", "machinations", "folly", "mythos", "fortune", "expansion", "edge", "undone", "winifred"
}))

In [120]:
## Create TfIdf vectorizer and fit_transform it to the comments
vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=10,
    stop_words=custom_stopwords,
    ngram_range=(1,1)
)
TfIdf_matrix = vectorizer.fit_transform(cleaned_docs)

## Create the NMF Model for topic analysis
model = NMF(n_components=3, random_state=42)
W = model.fit_transform(TfIdf_matrix)
H = model.components_

In [121]:
## Get the top keywords in each topic
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic #{topic_idx + 1}: {' | '.join(top_words)}")

Topic #1: time | story | rules | scenario | campaign | long | combat | mechanics | card | setup
Topic #2: cycle | campaign | core | starter | complete | decks | deluxe | standalone | packs | box
Topic #3: expansions | token | insert | base | scenarios | solo | pack | card | removable | box


# Topic Keywords
## Topic 1
 - time | story | rules | scenario | campaign | long | combat | mechanics | card | setup
## Topic 2
 - cycle | campaign | core | starter | complete | decks | deluxe | standalone | packs | box
## Topic 3
 - expansions | token | insert | base | scenarios | solo | pack | card | removable | box

# Topic Labels
- Pacing, Complexity, & Story Disappointment
- Expansion Dependency & Product Model Frustration
- Storage & Component Frustrations

# Strategic Questions

| **Question**                                                                 | **What to Do**                                                                  |
| ---------------------------------------------------------------------------- | ------------------------------------------------------------------------------- |
| What are the most common friction points players face with campaign games?   | Analyze reviews with high weights on the *pacing & story disappointment* topic. |
| How often do players feel forced to buy expansions to get a full experience? | Track frequency of the *expansion dependency* topic in core game reviews.       |
| What physical design choices lead to long-term dissatisfaction?              | Extract high-weight reviews from the *storage & component frustration* topic.   |